In [ ]:
!pip install transformers datasets accelerate pandas scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive is connected")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive is connected


In [ ]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorForSeq2Seq,
)
from sklearn.model_selection import train_test_split
import numpy as np


device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    gpu_props = torch.cuda.get_device_properties(0)
    print(f"GPU found: {gpu_props.name}, VRAM: {gpu_props.total_memory / 1024**3:.2f} GB")
else:
    print("No GPU found, using CPU.")

GPU found: Tesla T4, VRAM: 14.74 GB


In [ ]:
# --------------------------------------------------
# Model
# --------------------------------------------------
model_name = "Turkish-NLP/t5-efficient-small-MLSUM-TR-fine-tuned"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/839k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/569M [00:00<?, ?B/s]

In [ ]:
# --------------------------------------------------
# Data
# --------------------------------------------------

DATA_PATH = "/content/drive/MyDrive/AiProject/Datasets/mergedDataset.csv"

df = pd.read_csv(DATA_PATH)[["article_text", "summary"]].dropna()

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Loaded {len(df)} samples.")

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

Loaded 9123 samples.


In [ ]:
# --------------------------------------------------
# Tokenization
# --------------------------------------------------
max_input_length = 512
max_target_length = 64

def preprocess(examples):
    model_inputs = tokenizer(
        examples["article_text"],
        max_length=max_input_length,
        truncation=True,
        padding="max_length",
    )

    labels = tokenizer(
        text_target=examples["summary"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length",
    )

    # Replace padding token id's with -100 to ignore in loss
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(preprocess, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(preprocess, batched=True, remove_columns=val_dataset.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

Map:   0%|          | 0/8210 [00:00<?, ? examples/s]

Map:   0%|          | 0/913 [00:00<?, ? examples/s]

In [ ]:
# --------------------------------------------------
# Training setup
# --------------------------------------------------

# Google Drive'a kaydetmek için çıkış dizinini güncelleyin
output_path = "/content/drive/MyDrive/AiProject/Models/t5_finetune_results_4"

training_args = TrainingArguments(
    output_dir=output_path,
    eval_strategy="steps",
    eval_steps=400,
    save_steps=400,
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    fp16=False,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

callbacks = [EarlyStoppingCallback(early_stopping_patience=2)]

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=callbacks,
)

/tmp/ipython-input-3550472795.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# --------------------------------------------------
# Training
# --------------------------------------------------
print("Training has been started")
trainer.train()
print("Training is finished")

import shutil
from google.colab import files

# Modelin kayıtlı olduğu klasör (Senin kodundaki output_path burası olmalı)
# Eğer Drive'a kaydetmediysen "/content/t5_finetune_results" gibi bir yerdedir.
# Burayı kendi output_path değişkeninle aynı yap.
source_dir = "/content/drive/MyDrive/AiProject/Models/t5_finetune_results"

# Çıktı dosyasının adı ve yeri (/content/model_arsivi.zip olacak)
output_filename = "/content/model_arsivi"

print("Model klasörü zipleniyor...")
# Klasörü zip yap
shutil.make_archive(output_filename, 'zip', source_dir)

print("İndirme işlemi başlatılıyor...")
# Tarayıcı indirmesini tetikle
files.download(output_filename + ".zip")

Training has been started


Step,Training Loss,Validation Loss
400,2.052100,1.630731
800,1.880100,1.577764
1200,1.837500,1.553550
1600,1.805300,1.525430
2000,1.775800,1.507783
2400,1.675600,1.504894
2800,1.791600,1.490886
3200,1.490800,1.493464
3600,1.764100,1.477319
4000,1.818700,1.479545


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Training is finished


In [ ]:
# --------------------------------------------------
# Save model
# --------------------------------------------------

final_model_path = "/content/drive/MyDrive/AiProject/Models/t5_finetuned_4"

trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f" Model saved to {final_model_path}")

# --------------------------------------------------
# Example inference
# --------------------------------------------------
text = df["cleaned_article"].iloc[0]
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
summary_ids = model.generate(**inputs, max_length=64, num_beams=4)
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Example summary:\n", summary)

NameError: name 'trainer' is not defined